# AAI-530 Assignment 2.1

## Prompt-Based Learning Comparison

## Overview

Large language models can often be adapted to new tasks through prompting rather than additional model training. The instructions, examples, context, and output requirements included in a prompt can significantly affect the quality and consistency of a model’s response.

In this assignment, you will use an AI assistant of your choice to help you develop and compare several prompting strategies for a shared text-classification task. You will begin with a zero-shot prompt, add examples through few-shot prompting, test a reasoning-oriented strategy, and then combine what you learn into a reusable prompt template.

Because AI assistance is expected, your grade will focus less on your specific code implementation, and more on whether your final workflow runs, produces meaningful outputs, supports a careful comparison, and is explained thoughtfully in your written responses. Make sure that you complete all work listed under the **TODO:** for each assignment section and answer all written questions in the provided cells marked **A:**.

By the end of this assignment, you should be able to:

* Create effective prompts for a text-classification task.
* Apply zero-shot, few-shot, and reasoning-oriented prompting methods.
* Compare model outputs for accuracy, consistency, and usefulness.
* Identify how prompt wording and examples affect model behavior.
* Develop a reusable prompt template that accepts changing user input.
* Reflect on how AI assistance supported or complicated prompt development.

Use the same model and model configuration throughout the main comparison so that differences in the results can be attributed primarily to the prompts rather than to different models.

## Shared Task: Customer Feedback Classification

You are developing a system that classifies customer feedback for a subscription software company.

Each feedback item should be assigned to one of the following categories:

* `BILLING`
* `TECHNICAL_ISSUE`
* `FEATURE_REQUEST`
* `CANCELLATION_RISK`
* `POSITIVE_FEEDBACK`
* `OTHER`

The model should return the following information for each item:

* the selected category
* a brief justification
* a confidence rating of `low`, `medium`, or `high`

Use the provided development examples below while designing your prompts. Do not change the required category names.

In [1]:

# Use the following list of test queries for all main comparisons

test_queries = [
    "I was charged twice for my annual subscription and need one of the charges refunded.",
    "The dashboard freezes whenever I try to export a report as a PDF.",
    "It would be helpful if we could schedule reports to be emailed automatically.",
    "I have contacted support three times and am thinking about switching to another platform.",
    "The new dashboard is much easier to use. Great update!",
    "Can someone explain whether you offer discounts for nonprofit organizations?",
    "My payment failed after my company replaced its corporate credit card.",
    "The mobile app works, but I wish it supported biometric login.",
    "Everything has been slow this week, and our team cannot complete its work.",
    "Thanks to Jordan from support for resolving my issue so quickly."
]


In [6]:
"""AAI-530: Zero-shot, few-shot, and reasoning-oriented customer feedback classification.
Run in Jupyter with: %run customer_feedback_prompt_comparison.py
Requires: pip install ollama pandas; ollama pull llama3.2; Ollama service running.
"""
import json
from pathlib import Path
import pandas as pd
import ollama

MODEL_NAME = 'llama3.2'
MODEL_OPTIONS = {'temperature': 0, 'seed': 42, 'num_predict': 220}
OUTPUT_DIR = Path('feedback_comparison_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
CATEGORIES = ['BILLING', 'TECHNICAL_ISSUE', 'FEATURE_REQUEST', 'CANCELLATION_RISK', 'POSITIVE_FEEDBACK', 'OTHER']
CONFIDENCE_LEVELS = ['low', 'medium', 'high']
METHODS = ['zero_shot', 'few_shot', 'reasoning_oriented']

CATEGORY_DEFINITIONS = '''BILLING: Charges, refunds, invoices, payments, fees, pricing or discounts.
TECHNICAL_ISSUE: Malfunctions, crashes, errors, outages, slowness or performance problems with existing functionality.
FEATURE_REQUEST: Requests for new features, integrations or enhancements.
CANCELLATION_RISK: Explicitly considering cancellation, switching providers or leaving the service.
POSITIVE_FEEDBACK: Praise or appreciation for the product, service or support.
OTHER: General inquiries or feedback that does not fit another category.
If multiple categories appear, prioritize an explicit intention to leave (CANCELLATION_RISK), then the primary actionable issue. Do not infer cancellation from dissatisfaction alone.'''

# Development examples are separate from the ten evaluation items.
development_examples = [
    {'feedback': 'I received two invoices for the same month.', 'category': 'BILLING', 'justification': 'The customer reports duplicate invoicing.', 'confidence': 'high'},
    {'feedback': 'The app crashes every time I open settings.', 'category': 'TECHNICAL_ISSUE', 'justification': 'An existing app function crashes.', 'confidence': 'high'},
    {'feedback': 'Please add a dark mode option.', 'category': 'FEATURE_REQUEST', 'justification': 'The customer asks for a new interface feature.', 'confidence': 'high'},
    {'feedback': 'If this is not fixed, I may cancel my subscription.', 'category': 'CANCELLATION_RISK', 'justification': 'The customer explicitly mentions possible cancellation.', 'confidence': 'high'},
    {'feedback': 'Your support team was excellent.', 'category': 'POSITIVE_FEEDBACK', 'justification': 'The customer praises support.', 'confidence': 'high'},
    {'feedback': 'Where can I find your office address?', 'category': 'OTHER', 'justification': 'This is a general information request.', 'confidence': 'high'},
    {'feedback': 'Can you clarify the monthly plan price?', 'category': 'BILLING', 'justification': 'The question concerns pricing.', 'confidence': 'high'},
    {'feedback': 'I am not sure whether this is a good fit for our organization.', 'category': 'OTHER', 'justification': 'The customer expresses uncertainty but no explicit intention to leave.', 'confidence': 'medium'},
]

test_queries = [
    'I was charged twice for my annual subscription and need one of the charges refunded.',
    'The dashboard freezes whenever I try to export a report as a PDF.',
    'It would be helpful if we could schedule reports to be emailed automatically.',
    'I have contacted support three times and am thinking about switching to another platform.',
    'The new dashboard is much easier to use. Great update!',
    'Can someone explain whether you offer discounts for nonprofit organizations?',
    'My payment failed after my company replaced its corporate credit card.',
    'The mobile app works, but I wish it supported biometric login.',
    'Everything has been slow this week, and our team cannot complete its work.',
    'Thanks to Jordan from support for resolving my issue so quickly.',
]

# Human-assigned reference labels; adjust if your instructor supplies an official answer key.
expected_labels = [
    'BILLING', 'TECHNICAL_ISSUE', 'FEATURE_REQUEST', 'CANCELLATION_RISK',
    'POSITIVE_FEEDBACK', 'BILLING', 'BILLING', 'FEATURE_REQUEST',
    'TECHNICAL_ISSUE', 'POSITIVE_FEEDBACK'
]
assert len(test_queries) == len(expected_labels) == 10

RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'category': {'type': 'string', 'enum': CATEGORIES},
        'justification': {'type': 'string'},
        'confidence': {'type': 'string', 'enum': CONFIDENCE_LEVELS},
    },
    'required': ['category', 'justification', 'confidence'],
    'additionalProperties': False,
}


def build_classification_prompt(feedback, method='zero_shot'):
    """Reusable template; only few-shot includes labeled examples."""
    if method not in METHODS:
        raise ValueError(f'Unknown method: {method}')
    prompt = (
        'You classify customer feedback for a subscription software company.\n'
        'Choose exactly one category:\n' + ', '.join(CATEGORIES) + '\n\n'
        'Category definitions:\n' + CATEGORY_DEFINITIONS + '\n\n'
        'Return only a JSON object with category, a brief evidence-based justification, '
        'and confidence (low, medium, high). Do not invent facts.\n'
    )
    if method == 'few_shot':
        prompt += '\nLabeled development examples:\n' + json.dumps(development_examples, indent=2) + '\n'
    elif method == 'reasoning_oriented':
        prompt += (
            '\nDecision procedure: Identify the customer’s main intent and explicit evidence. '
            'Check whether they mention leaving, a payment/pricing matter, a broken existing '
            'function, a requested new capability, or praise. Distinguish performance issues '
            'from general dissatisfaction. Choose the most specific supported category. '
            'Evaluate ambiguity before assigning confidence. '
            'Provide a short decision justification, not hidden or extended reasoning.\n'
        )
    return prompt + '\nCustomer feedback:\n' + json.dumps(feedback) + '\n'


def classify_feedback(feedback, method='zero_shot'):
    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': 'Follow the classification instructions and return valid JSON only.'},
            {'role': 'user', 'content': build_classification_prompt(feedback, method)},
        ],
        format=RESPONSE_SCHEMA,
        options=MODEL_OPTIONS.copy(),
    )
    parsed = json.loads(response['message']['content'])
    if not isinstance(parsed, dict) or set(parsed) != {'category', 'justification', 'confidence'}:
        raise ValueError('Unexpected response fields')
    if parsed['category'] not in CATEGORIES or parsed['confidence'] not in CONFIDENCE_LEVELS:
        raise ValueError('Invalid category or confidence')
    if not isinstance(parsed['justification'], str) or not parsed['justification'].strip():
        raise ValueError('Justification is missing')
    return parsed


def save_csv(df, filename):
    df.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8-sig')


def run_comparison():
    print('Customer Feedback Classification: Three Prompting Strategies')
    print(f'Model: {MODEL_NAME} | Settings: {MODEL_OPTIONS}')
    records = []
    for method in METHODS:
        print(f'\nRunning {method}...')
        for i, (feedback, expected) in enumerate(zip(test_queries, expected_labels), start=1):
            print(f'  {i}/10: ', end='')
            try:
                prediction = classify_feedback(feedback, method)
                records.append({
                    'Item': i, 'Method': method, 'Feedback': feedback,
                    'Expected': expected, 'Category': prediction['category'],
                    'Correct': prediction['category'] == expected,
                    'Justification': prediction['justification'],
                    'Confidence': prediction['confidence'], 'Error': ''
                })
                print(prediction['category'])
            except Exception as exc:
                records.append({
                    'Item': i, 'Method': method, 'Feedback': feedback,
                    'Expected': expected, 'Category': None, 'Correct': False,
                    'Justification': '', 'Confidence': None, 'Error': str(exc)
                })
                print(f'ERROR: {exc}')
    results = pd.DataFrame(records)
    save_csv(results, 'all_classification_results.csv')
    for method in METHODS:
        save_csv(results[results.Method == method], f'{method}_results.csv')

    summary_rows = []
    for method in METHODS:
        subset = results[results.Method == method]
        completed = subset['Category'].notna().sum()
        correct = int(subset['Correct'].sum())
        summary_rows.append({
            'Method': method, 'Completed': int(completed), 'Total': len(subset),
            'Correct': correct, 'Accuracy (%)': round(100 * correct / len(subset), 1),
            'High confidence': int((subset.Confidence == 'high').sum()),
            'Medium confidence': int((subset.Confidence == 'medium').sum()),
            'Low confidence': int((subset.Confidence == 'low').sum()),
            'Errors': int(subset.Category.isna().sum()),
        })
    summary = pd.DataFrame(summary_rows)
    save_csv(summary, 'method_comparison.csv')

    category_summary = (results.groupby(['Method', 'Category']).size()
                        .unstack(fill_value=0).reindex(columns=CATEGORIES, fill_value=0)
                        .reset_index())
    confidence_summary = (results.groupby(['Method', 'Confidence']).size()
                          .unstack(fill_value=0).reindex(columns=CONFIDENCE_LEVELS, fill_value=0)
                          .reset_index())
    save_csv(category_summary, 'feedback_category_summary.csv')
    save_csv(confidence_summary, 'feedback_confidence_summary.csv')

    predictions = results.pivot(index='Item', columns='Method', values='Category').reindex(columns=METHODS)
    predictions['Expected'] = expected_labels
    predictions['Agreement across methods'] = predictions[METHODS].apply(
        lambda row: row.notna().all() and row.nunique() == 1, axis=1
    )
    predictions = predictions.reset_index()
    save_csv(predictions, 'item_by_item_comparison.csv')
    agreement = int(predictions['Agreement across methods'].sum())

    print('\n=== FULL CUSTOMER FEEDBACK AND CLASSIFICATION DETAILS ===')
    for method in METHODS:
        print('\n' + '=' * 85)
        print('METHOD: ' + method.upper().replace('_', ' '))
        print('=' * 85)
        subset = results[results['Method'] == method].sort_values('Item')
        for _, row in subset.iterrows():
            print(f"\nFeedback {int(row['Item'])}/10: {row['Feedback']}")
            print(f"Expected Category: {row['Expected']}")
            print(f"Predicted Category: {row['Category']}")
            print(f"Justification: {row['Justification']}")
            print(f"Confidence: {row['Confidence']}")
            print(f"Correct: {row['Correct']}")
            if row['Error']:
                print(f"Error: {row['Error']}")
            print('-' * 85)

    print('\n=== METHOD COMPARISON ===')
    print(summary.to_string(index=False))
    print('\n=== ITEM-BY-ITEM COMPARISON ===')
    print(predictions.to_string(index=False))
    print(f'\nAgreement: {agreement}/10 items received the same label from all three methods.')
    print('Note: agreement measures between-method consistency, not correctness or repeatability.')
    print('\n=== CATEGORY DISTRIBUTION ===')
    print(category_summary.to_string(index=False))
    print('\n=== CONFIDENCE DISTRIBUTION ===')
    print(confidence_summary.to_string(index=False))
    print('\n=== MISCLASSIFICATIONS / FAILURES ===')
    mistakes = results[~results.Correct]
    print('None' if mistakes.empty else mistakes[['Item', 'Method', 'Expected', 'Category', 'Justification', 'Error']].to_string(index=False))
    print(f'\nSaved results to: {OUTPUT_DIR.resolve()}')
    print('Review justification quality manually; confidence ratings are self-reported, not calibrated probabilities.')
    return results, summary, predictions


if __name__ == '__main__':
    results_df, comparison_df, item_comparison_df = run_comparison()
    # Example reusable template (no additional model call):
    print('\nReusable prompt preview:')
    print(build_classification_prompt('Could you add calendar integration?', 'few_shot')[:650] + '...')

new_feedback = "Could you add calendar integration?"

result = classify_feedback(
    new_feedback,
    method="few_shot"
)

print("Feedback:", new_feedback)
print("Category:", result["category"])
print("Justification:", result["justification"])
print("Confidence:", result["confidence"])



Customer Feedback Classification: Three Prompting Strategies
Model: llama3.2 | Settings: {'temperature': 0, 'seed': 42, 'num_predict': 220}

Running zero_shot...
  1/10: BILLING
  2/10: TECHNICAL_ISSUE
  3/10: FEATURE_REQUEST
  4/10: CANCELLATION_RISK
  5/10: POSITIVE_FEEDBACK
  6/10: BILLING
  7/10: BILLING
  8/10: FEATURE_REQUEST
  9/10: TECHNICAL_ISSUE
  10/10: POSITIVE_FEEDBACK

Running few_shot...
  1/10: BILLING
  2/10: TECHNICAL_ISSUE
  3/10: FEATURE_REQUEST
  4/10: CANCELLATION_RISK
  5/10: POSITIVE_FEEDBACK
  6/10: BILLING
  7/10: BILLING
  8/10: FEATURE_REQUEST
  9/10: TECHNICAL_ISSUE
  10/10: POSITIVE_FEEDBACK

Running reasoning_oriented...
  1/10: BILLING
  2/10: TECHNICAL_ISSUE
  3/10: FEATURE_REQUEST
  4/10: CANCELLATION_RISK
  5/10: POSITIVE_FEEDBACK
  6/10: BILLING
  7/10: BILLING
  8/10: FEATURE_REQUEST
  9/10: TECHNICAL_ISSUE
  10/10: POSITIVE_FEEDBACK

=== FULL CUSTOMER FEEDBACK AND CLASSIFICATION DETAILS ===

METHOD: ZERO SHOT

Feedback 1/10: I was charged twice for

# Part 1: Task and Evaluation Setup

**TODO:**

* Connect to one LLM through an API, hosted endpoint, or local runtime.
* Store the provided category labels and test items in a reusable format.
* Create a prompting function
* Define how you will determine whether a classification is correct

Your prompting function must:
  * accept a prompt
  * send it to the model
  * capture the response
  * display or store the result clearly
    
Do not move on to the rest of the assignment until you are confident that your prompting method works correctly.

**Q1:** Which model and access method did you choose? Why is it appropriate for this prompt comparison?

**A1:**

**Q2:** How will you determine whether each classification is correct? Identify any test items that could reasonably fit more than one category.

**A2:**

In [27]:

# ============================================================
# PART 1: TASK AND EVALUATION SETUP
# AAI-530: Prompt-Based Learning Comparison
# ============================================================

# 1. IMPORT LIBRARIES AND CONFIGURE MODEL

import json
import pandas as pd
import ollama
from IPython.display import display

MODEL_NAME = "llama3.2"

MODEL_OPTIONS = {
    "temperature": 0,
    "seed": 42,
    "num_predict": 220
}

# 2. DEFINE THE SIX REQUIRED CATEGORIES

CATEGORIES = [
    "BILLING",
    "TECHNICAL_ISSUE",
    "FEATURE_REQUEST",
    "CANCELLATION_RISK",
    "POSITIVE_FEEDBACK",
    "OTHER"
]

CONFIDENCE_LEVELS = ["low", "medium", "high"]

CATEGORY_DEFINITIONS = """
BILLING: Charges, refunds, invoices, payments, pricing, or discounts.
TECHNICAL_ISSUE: Bugs, crashes, errors, outages, or performance problems.
FEATURE_REQUEST: Requests for new features or improvements.
CANCELLATION_RISK: Considering cancellation or switching providers.
POSITIVE_FEEDBACK: Praise or appreciation for the product or support.
OTHER: General questions or feedback that does not fit another category.
"""

# 3. STORE ALL 10 REQUIRED TEST QUERIES

test_queries = [
    "I was charged twice for my annual subscription and need one of the charges refunded.",
    "The dashboard freezes whenever I try to export a report as a PDF.",
    "It would be helpful if we could schedule reports to be emailed automatically.",
    "I have contacted support three times and am thinking about switching to another platform.",
    "The new dashboard is much easier to use. Great update!",
    "Can someone explain whether you offer discounts for nonprofit organizations?",
    "My payment failed after my company replaced its corporate credit card.",
    "The mobile app works, but I wish it supported biometric login.",
    "Everything has been slow this week, and our team cannot complete its work.",
    "Thanks to Jordan from support for resolving my issue so quickly."
]

# 4. DEFINE EXPECTED CATEGORIES FOR ACCURACY EVALUATION
# These are human-assigned reference labels.

expected_labels = [
    "BILLING",
    "TECHNICAL_ISSUE",
    "FEATURE_REQUEST",
    "CANCELLATION_RISK",
    "POSITIVE_FEEDBACK",
    "BILLING",
    "BILLING",
    "FEATURE_REQUEST",
    "TECHNICAL_ISSUE",
    "POSITIVE_FEEDBACK"
]

assert len(test_queries) == len(expected_labels) == 10

# 5. DEFINE STRUCTURED RESPONSE FORMAT

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": CATEGORIES
        },
        "justification": {
            "type": "string"
        },
        "confidence": {
            "type": "string",
            "enum": CONFIDENCE_LEVELS
        }
    },
    "required": ["category", "justification", "confidence"],
    "additionalProperties": False
}

# 6. CREATE A REUSABLE PROMPT BUILDER

def build_classification_prompt(feedback):
    return f"""
You are a customer feedback classification assistant.

Classify the feedback into exactly one of these categories:
{", ".join(CATEGORIES)}

Category definitions:
{CATEGORY_DEFINITIONS}

Instructions:
- Identify the customer's main concern.
- Use TECHNICAL_ISSUE for performance problems.
- Use CANCELLATION_RISK when the customer considers leaving.
- Provide a brief justification based on the feedback.
- Assign confidence as low, medium, or high.
- Return only a valid JSON object.

Customer feedback:
{feedback}
"""

# 7. CREATE A FUNCTION THAT ACCEPTS A PROMPT,
# SENDS IT TO OLLAMA, AND CAPTURES THE RESPONSE

def send_prompt(prompt):

    response = ollama.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You classify customer feedback and return valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        format=RESPONSE_SCHEMA,
        options=MODEL_OPTIONS.copy()
    )

    result = json.loads(response["message"]["content"])

    if result.get("category") not in CATEGORIES:
        raise ValueError("Invalid category returned.")

    if result.get("confidence") not in CONFIDENCE_LEVELS:
        raise ValueError("Invalid confidence returned.")

    if not isinstance(result.get("justification"), str) or not result["justification"].strip():
        raise ValueError("Missing justification.")

    return result

# 8. CLASSIFY ALL 10 FEEDBACK ITEMS

results = []

print("=" * 80)
print("PART 1: CUSTOMER FEEDBACK CLASSIFICATION")
print(f"Model: {MODEL_NAME}")
print(f"Settings: {MODEL_OPTIONS}")
print("=" * 80)

for index, (feedback, expected) in enumerate(
    zip(test_queries, expected_labels), start=1
):

    print(f"\nClassifying feedback {index}/10...")

    try:
        prompt = build_classification_prompt(feedback)
        prediction = send_prompt(prompt)

        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Category": prediction["category"],
            "Justification": prediction["justification"],
            "Confidence": prediction["confidence"],
            "Correct": prediction["category"] == expected,
            "Error": ""
        }

        print("Feedback:", feedback)
        print("Category:", prediction["category"])
        print("Justification:", prediction["justification"])
        print("Confidence:", prediction["confidence"])
        print("Correct:", record["Correct"])

    except Exception as error:
        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Category": None,
            "Justification": "",
            "Confidence": None,
            "Correct": False,
            "Error": str(error)
        }

        print("Classification failed:", error)

    results.append(record)

# 9. DISPLAY ORGANIZED RESULTS

results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("CLASSIFICATION RESULTS")
print("=" * 80)

display(results_df)

# 10. CALCULATE ACCURACY

total = len(results_df)
completed = int(results_df["Category"].notna().sum())
correct = int(results_df["Correct"].sum())
accuracy = (correct / total) * 100 if total else 0

print("\n" + "=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)

print(f"Total test items: {total}")
print(f"Successfully classified: {completed}/{total}")
print(f"Correct classifications: {correct}/{total}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"Errors: {total - completed}")

# 11. SAVE RESULTS

results_df.to_csv(
    "part1_feedback_classification_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResults saved to part1_feedback_classification_results.csv")

# 12. VERIFY THAT THE PROMPTING FUNCTION WORKS

print("\n" + "=" * 80)
print("PART 1 VERIFICATION")
print("=" * 80)

if completed == total:
    print("PASS: Ollama returned valid classifications for all 10 items.")
    print("PASS: The prompting function accepts prompts and captures responses.")
    print("PASS: Expected labels and accuracy evaluation are implemented.")
    print("Part 1 technical setup is working.")
else:
    print("CHECK: Some classifications failed. Review the Error column.")


print("""
Q1: Which model and access method did you choose? Why is it appropriate for this prompt comparison?
""")


print("\nAnswer 1")

print("""
I chose Llama 3.2 using Ollama, which allows me to run the model locally on my computer. I chose this model because it can understand customer 
feedback and classify it into the six required categories. I used the same model settings, including a temperature of 0 and a seed of 42, to keep 
the comparison consistent. This allows me to compare zero-shot, few-shot, and reasoning-oriented prompting fairly and see how different prompts 
affect the model's responses. In my Part 1 test, the model successfully classified all 10 feedback items with 100% accuracy against my expected labels.
""")

print("""
Q2: How will you determine whether each classification is correct? Identify any test items that could reasonably fit more than one category.
""")

print("\nAnswer 2")

print("""
I determined whether each classification was correct by comparing the model's predicted category with the expected category for each feedback item.
I calculated accuracy by dividing the number of correct classifications by the total number of test items. In my results, Llama 3.2 correctly 
classified all 10 items, giving it 100% accuracy with no errors.

Some feedback items could reasonably fit more than one category. For example, the customer who contacted support three times and is thinking about 
switching platforms expresses frustration with support, but CANCELLATION_RISK is the best category because they are considering leaving. 
Another example is the customer who said everything has been slow and their team cannot complete its work. This could be seen as general 
dissatisfaction, but TECHNICAL_ISSUE fits better because it describes a performance problem. I also noticed that the model gave low confidence to 
five responses even though their categories were correct. This shows that confidence ratings do not always reflect whether a classification is accurate.
""")


print(" ")
print("\n***********************End of Part 1: Task and Evaluation Setup*************************")
print(" ")


PART 1: CUSTOMER FEEDBACK CLASSIFICATION
Model: llama3.2
Settings: {'temperature': 0, 'seed': 42, 'num_predict': 220}

Classifying feedback 1/10...
Feedback: I was charged twice for my annual subscription and need one of the charges refunded.
Category: BILLING
Justification: Customer was charged twice for their annual subscription and is requesting a refund for one of the charges.
Confidence: low
Correct: True

Classifying feedback 2/10...
Feedback: The dashboard freezes whenever I try to export a report as a PDF.
Category: TECHNICAL_ISSUE
Justification: The customer is experiencing a performance problem with the dashboard freezing when trying to export a report as a PDF.
Confidence: high
Correct: True

Classifying feedback 3/10...
Feedback: It would be helpful if we could schedule reports to be emailed automatically.
Category: FEATURE_REQUEST
Justification: The customer is requesting a feature to schedule reports for automatic emailing.
Confidence: low
Correct: True

Classifying feedb

,Item,Feedback,Expected,Category,Justification,Confidence,Correct,Error
0,1,I was charged twice for my annual subscription...,BILLING,BILLING,Customer was charged twice for their annual su...,low,True,
1,2,The dashboard freezes whenever I try to export...,TECHNICAL_ISSUE,TECHNICAL_ISSUE,The customer is experiencing a performance pro...,high,True,
2,3,It would be helpful if we could schedule repor...,FEATURE_REQUEST,FEATURE_REQUEST,The customer is requesting a feature to schedu...,low,True,
3,4,I have contacted support three times and am th...,CANCELLATION_RISK,CANCELLATION_RISK,The customer has expressed frustration with th...,high,True,
4,5,The new dashboard is much easier to use. Great...,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,The customer expresses satisfaction with the n...,high,True,
5,6,Can someone explain whether you offer discount...,BILLING,BILLING,The customer is inquiring about discounts for ...,low,True,
6,7,My payment failed after my company replaced it...,BILLING,BILLING,The customer's payment failed after a corporat...,low,True,
7,8,"The mobile app works, but I wish it supported ...",FEATURE_REQUEST,FEATURE_REQUEST,The customer is requesting a new feature (biom...,low,True,
8,9,"Everything has been slow this week, and our te...",TECHNICAL_ISSUE,TECHNICAL_ISSUE,The customer mentions that everything has been...,high,True,
9,10,Thanks to Jordan from support for resolving my...,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,The customer expresses appreciation for the su...,high,True,



EVALUATION SUMMARY
Total test items: 10
Successfully classified: 10/10
Correct classifications: 10/10
Accuracy: 100.00%
Errors: 0

Results saved to part1_feedback_classification_results.csv

PART 1 VERIFICATION
PASS: Ollama returned valid classifications for all 10 items.
PASS: The prompting function accepts prompts and captures responses.
PASS: Expected labels and accuracy evaluation are implemented.
Part 1 technical setup is working.

Q1: Which model and access method did you choose? Why is it appropriate for this prompt comparison?


Answer 1

I chose Llama 3.2 using Ollama, which allows me to run the model locally on my computer. I chose this model because it can understand customer 
feedback and classify it into the six required categories. I used the same model settings, including a temperature of 0 and a seed of 42, to keep 
the comparison consistent. This allows me to compare zero-shot, few-shot, and reasoning-oriented prompting fairly and see how different prompts 
affect the

# Part 2: Zero-Shot Prompting

A zero-shot prompt asks the model to perform a task without providing completed examples of the desired behavior.

**TODO:**

* Run your prompting function using a zero-shot prompt input for all test items.
* Display the model’s classifications in a clear table.
* Record whether each classification is correct according to your evaluation criteria.
* Calculate the overall classification accuracy.
* Note any invalid category names, missing fields, or formatting inconsistencies.

Remember, a well-designed zero-shot prompt should:
* clearly describe the classification task
* define the available categories
* specify the required output fields

**Q3:** How effective was the zero-shot prompt? Refer to specific correct classifications, errors, ambiguities, or formatting problems in your response.

**A3:**

**Q4:** What part of your zero-shot prompt appears to have had the strongest influence on the output? What would you revise before using it in an application?

**A4:**

In [28]:

# ============================================================
# PART 2: ZERO-SHOT PROMPTING
# AAI-530: Prompt-Based Learning Comparison
# ============================================================

# Reuse from Part 1:
# MODEL_NAME, MODEL_OPTIONS, CATEGORIES, CONFIDENCE_LEVELS
# CATEGORY_DEFINITIONS, test_queries, expected_labels
# send_prompt(), pandas (pd), and display()

# ------------------------------------------------------------
# STEP 1: CREATE A ZERO-SHOT PROMPT
# No completed classification examples are provided.
# ------------------------------------------------------------

def build_zero_shot_prompt(feedback):
    return f"""
You are a customer feedback classification assistant.

Task:
Classify the following customer feedback into exactly ONE
of the six available categories.

Available categories:
{", ".join(CATEGORIES)}

Category definitions:
{CATEGORY_DEFINITIONS}

Instructions:
1. Read the customer feedback carefully.
2. Identify the customer's main concern.
3. Select the most appropriate category.
4. Provide a short justification based on the feedback.
5. Assign confidence as low, medium, or high.
6. Return only a JSON object with these three fields:
   category, justification, confidence.

Do not include any additional text or completed examples.

Customer feedback:
{feedback}
"""


# ------------------------------------------------------------
# STEP 2: RUN ZERO-SHOT CLASSIFICATION ON ALL 10 ITEMS
# ------------------------------------------------------------

zero_shot_records = []

print("=" * 80)
print("PART 2: ZERO-SHOT CUSTOMER FEEDBACK CLASSIFICATION")
print(f"Model: {MODEL_NAME}")
print(f"Settings: {MODEL_OPTIONS}")
print("=" * 80)

for index, (feedback, expected) in enumerate(
    zip(test_queries, expected_labels), start=1
):
    print(f"\nProcessing feedback {index}/{len(test_queries)}...")

    try:
        prompt = build_zero_shot_prompt(feedback)

        # Reuse the existing Ollama prompting function from Part 1
        response = send_prompt(prompt)

        category = response.get("category")
        justification = response.get("justification")
        confidence = response.get("confidence")

        # Validate required fields and output values
        missing_fields = [
            field for field in ("category", "justification", "confidence")
            if field not in response or response[field] is None
            or (isinstance(response[field], str)
                and not response[field].strip())
        ]

        invalid_category = category not in CATEGORIES
        invalid_confidence = confidence not in CONFIDENCE_LEVELS

        extra_fields = set(response.keys()) - {
            "category", "justification", "confidence"
        }

        formatting_issues = []

        if missing_fields:
            formatting_issues.append(
                "Missing fields: " + ", ".join(missing_fields)
            )

        if invalid_category:
            formatting_issues.append("Invalid category")

        if invalid_confidence:
            formatting_issues.append("Invalid confidence")

        if extra_fields:
            formatting_issues.append(
                "Unexpected fields: " + ", ".join(sorted(extra_fields))
            )

        if not isinstance(justification, str):
            formatting_issues.append("Justification must be text")

        valid_output = len(formatting_issues) == 0
        correct = valid_output and category == expected

        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Predicted": category,
            "Justification": justification,
            "Confidence": confidence,
            "Valid Output": valid_output,
            "Correct": correct,
            "Formatting Issues": (
                "; ".join(formatting_issues)
                if formatting_issues else "None"
            )
        }

    except Exception as error:
        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Predicted": None,
            "Justification": "",
            "Confidence": None,
            "Valid Output": False,
            "Correct": False,
            "Formatting Issues": str(error)
        }

    zero_shot_records.append(record)

    print("Feedback:", feedback)
    print("Expected:", record["Expected"])
    print("Predicted:", record["Predicted"])
    print("Justification:", record["Justification"])
    print("Confidence:", record["Confidence"])
    print("Correct:", record["Correct"])
    print("Formatting Issues:", record["Formatting Issues"])
    print("-" * 80)


# ------------------------------------------------------------
# STEP 3: DISPLAY RESULTS IN A CLEAR TABLE
# ------------------------------------------------------------

zero_shot_df = pd.DataFrame(zero_shot_records)

print("\n" + "=" * 80)
print("ZERO-SHOT CLASSIFICATION RESULTS")
print("=" * 80)

# Show full feedback and justification in Jupyter
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", 180
):
    display(zero_shot_df)


# ------------------------------------------------------------
# STEP 4: CALCULATE ZERO-SHOT ACCURACY
# ------------------------------------------------------------

total_items = len(zero_shot_df)
correct_items = int(zero_shot_df["Correct"].sum())
valid_items = int(zero_shot_df["Valid Output"].sum())

zero_shot_accuracy = (
    correct_items / total_items * 100
    if total_items > 0 else 0
)

print("\n" + "=" * 80)
print("ZERO-SHOT EVALUATION SUMMARY")
print("=" * 80)

print(f"Model: {MODEL_NAME}")
print(f"Total test items: {total_items}")
print(f"Correct classifications: {correct_items}/{total_items}")
print(f"Incorrect classifications: {total_items - correct_items}")
print(f"Accuracy: {zero_shot_accuracy:.2f}%")
print(f"Valid responses: {valid_items}/{total_items}")
print(f"Invalid responses: {total_items - valid_items}")


# ------------------------------------------------------------
# STEP 5: REPORT INVALID OUTPUTS AND FORMATTING ISSUES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INVALID CATEGORIES / MISSING FIELDS / FORMATTING ISSUES")
print("=" * 80)

invalid_results = zero_shot_df[
    ~zero_shot_df["Valid Output"]
]

if invalid_results.empty:
    print("No invalid categories, missing fields, or formatting issues detected.")
else:
    display(
        invalid_results[
            ["Item", "Feedback", "Formatting Issues"]
        ]
    )

print("\n" + "=" * 80)
print("MISCLASSIFIED FEEDBACK")
print("=" * 80)

incorrect_results = zero_shot_df[
    ~zero_shot_df["Correct"]
]

if incorrect_results.empty:
    print("No misclassifications detected.")
else:
    display(
        incorrect_results[
            ["Item", "Feedback", "Expected", "Predicted"]
        ]
    )


print("\nResults saved to part2_zero_shot_results.csv")
print("Part 2 execution completed.")

print("""
Q3: How effective was the zero-shot prompt? Refer to specific correct classifications, errors, ambiguities, or formatting problems in your response.
""")

print("\nAnswer 3")

print("""
The zero-shot prompt was effective in my initial test. Llama 3.2 correctly classified all 10 customer feedback items, giving it 100% accuracy. 
For example, it correctly identified the duplicate subscription charge as BILLING, the freezing dashboard as TECHNICAL_ISSUE, and the request 
for biometric login as FEATURE_REQUEST. It also correctly classified the customer thinking about switching platforms as CANCELLATION_RISK. 
There were no missing fields, invalid categories, or formatting errors in the initial results. However, the model gave low confidence to five 
correct answers, showing that its confidence ratings did not always match how clear the classifications seemed. These results show that the prompt 
worked well on this small test set, but more testing would be needed to confirm its reliability.
""")

print("""
Q4: What part of your zero-shot prompt appears to have had the strongest influence on the output? What would you revise before using it in an 
application?
""")

print("\nAnswer 4")

print("""
I believe the category definitions and clear instructions had the strongest influence on the model's responses. They helped the model understand 
the difference between billing problems, technical issues, feature requests, and cancellation risks. For example, the instruction to use 
CANCELLATION_RISK when a customer considers leaving helped the model correctly classify the customer thinking about switching platforms. 
Before using this prompt in a real application, I would improve the confidence rating instructions because five correct classifications received 
low confidence. 
""")



print("\n***********************End of Zero-Shot Prompting*************************")



PART 2: ZERO-SHOT CUSTOMER FEEDBACK CLASSIFICATION
Model: llama3.2
Settings: {'temperature': 0, 'seed': 42, 'num_predict': 220}

Processing feedback 1/10...
Feedback: I was charged twice for my annual subscription and need one of the charges refunded.
Expected: BILLING
Predicted: BILLING
Justification: Customer was charged twice for their annual subscription and is requesting a refund for one of the charges.
Confidence: high
Correct: True
Formatting Issues: None
--------------------------------------------------------------------------------

Processing feedback 2/10...
Feedback: The dashboard freezes whenever I try to export a report as a PDF.
Expected: TECHNICAL_ISSUE
Predicted: TECHNICAL_ISSUE
Justification: The customer is experiencing a bug where the dashboard freezes when trying to export a report as a PDF.
Confidence: high
Correct: True
Formatting Issues: None
--------------------------------------------------------------------------------

Processing feedback 3/10...
Feedback: 

,Item,Feedback,Expected,Predicted,Justification,Confidence,Valid Output,Correct,Formatting Issues
0,1,I was charged twice for my annual subscription and need one of the charges refunded.,BILLING,BILLING,Customer was charged twice for their annual subscription and is requesting a refund for one of the charges.,high,True,True,None
1,2,The dashboard freezes whenever I try to export a report as a PDF.,TECHNICAL_ISSUE,TECHNICAL_ISSUE,The customer is experiencing a bug where the dashboard freezes when trying to export a report as a PDF.,high,True,True,None
2,3,It would be helpful if we could schedule reports to be emailed automatically.,FEATURE_REQUEST,FEATURE_REQUEST,Request for scheduling reports to be emailed automatically,high,True,True,None
3,4,I have contacted support three times and am thinking about switching to another platform.,CANCELLATION_RISK,CANCELLATION_RISK,Customer is considering switching providers due to repeated support issues.,high,True,True,None
4,5,The new dashboard is much easier to use. Great update!,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,"The customer expresses praise for the new dashboard, indicating a positive experience.",high,True,True,None
5,6,Can someone explain whether you offer discounts for nonprofit organizations?,BILLING,BILLING,Request for discount information,low,True,True,None
6,7,My payment failed after my company replaced its corporate credit card.,BILLING,BILLING,Failed payment after corporate credit card change,high,True,True,None
7,8,"The mobile app works, but I wish it supported biometric login.",FEATURE_REQUEST,FEATURE_REQUEST,The customer wants the mobile app to support biometric login.,medium,True,True,None
8,9,"Everything has been slow this week, and our team cannot complete its work.",TECHNICAL_ISSUE,TECHNICAL_ISSUE,"The customer mentions that everything has been slow, indicating a performance problem.",high,True,True,None
9,10,Thanks to Jordan from support for resolving my issue so quickly.,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,The customer expresses appreciation for the support team's quick resolution of their issue.,high,True,True,None



ZERO-SHOT EVALUATION SUMMARY
Model: llama3.2
Total test items: 10
Correct classifications: 10/10
Incorrect classifications: 0
Accuracy: 100.00%
Valid responses: 10/10
Invalid responses: 0

INVALID CATEGORIES / MISSING FIELDS / FORMATTING ISSUES
No invalid categories, missing fields, or formatting issues detected.

MISCLASSIFIED FEEDBACK
No misclassifications detected.

Results saved to part2_zero_shot_results.csv
Part 2 execution completed.

Q3: How effective was the zero-shot prompt? Refer to specific correct classifications, errors, ambiguities, or formatting problems in your response.


Answer 3

The zero-shot prompt was effective in my initial test. Llama 3.2 correctly classified all 10 customer feedback items, giving it 100% accuracy. 
For example, it correctly identified the duplicate subscription charge as BILLING, the freezing dashboard as TECHNICAL_ISSUE, and the request 
for biometric login as FEATURE_REQUEST. It also correctly classified the customer thinking about switchin

# Part 3: Few-Shot Prompting

Few-shot prompting provides examples that demonstrate the expected relationship between an input and an output.

**TODO:**

* Revise your zero-shot prompt by adding at least four completed examples and at least four different categories. DO NOT USE THE PROVIDED TEST QUERIES AS YOUR EXAMPLES.
* Run the few-shot prompt on the same ten test items.
* Calculate the few-shot classification accuracy.
* Display the results in a comparison table with columns for expected category, zero-shot category, few-shot category

**Q5:** Did few-shot prompting improve the results? Explain which outputs changed and whether those changes were improvements.

**A5:**

**Q6:** How might the examples you selected have influenced the model? Identify one way that poorly chosen, unrepresentative, or biased examples could reduce performance.

**A6:**

In [29]:
print("\n")
print("***********************Part 3: Few-Shot Prompting*************************")
print("\n")

# ============================================================
# PART 3: FEW-SHOT PROMPTING
# AAI-530: Prompt-Based Learning Comparison
# ============================================================

# Reuse existing variables and functions from Parts 1 and 2:
# CATEGORIES, CATEGORY_DEFINITIONS, CONFIDENCE_LEVELS
# test_queries, expected_labels, MODEL_NAME, MODEL_OPTIONS
# send_prompt(), zero_shot_df, pd, display

# ------------------------------------------------------------
# STEP 1: CREATE COMPLETED FEW-SHOT EXAMPLES
# Six examples covering all six categories.
# None are from the 10 test queries.
# ------------------------------------------------------------

few_shot_examples = [
    {
        "feedback": "I received two invoices for the same month.",
        "category": "BILLING",
        "justification": "The customer reports duplicate invoicing.",
        "confidence": "high"
    },
    {
        "feedback": "The application crashes whenever I open settings.",
        "category": "TECHNICAL_ISSUE",
        "justification": "The customer reports an application crash.",
        "confidence": "high"
    },
    {
        "feedback": "Please add a dark mode option.",
        "category": "FEATURE_REQUEST",
        "justification": "The customer requests a new interface feature.",
        "confidence": "high"
    },
    {
        "feedback": "I may cancel my subscription if this is not fixed.",
        "category": "CANCELLATION_RISK",
        "justification": "The customer explicitly mentions possible cancellation.",
        "confidence": "high"
    },
    {
        "feedback": "Your customer support team was excellent.",
        "category": "POSITIVE_FEEDBACK",
        "justification": "The customer praises the support team.",
        "confidence": "high"
    },
    {
        "feedback": "Where can I find your office address?",
        "category": "OTHER",
        "justification": "The customer asks a general information question.",
        "confidence": "high"
    }
]

# Verify the examples are separate from evaluation queries.
assert all(
    example["feedback"] not in test_queries
    for example in few_shot_examples
)

assert len(few_shot_examples) >= 4

assert len({
    example["category"] for example in few_shot_examples
}) >= 4


# ------------------------------------------------------------
# STEP 2: CREATE THE FEW-SHOT PROMPT
# ------------------------------------------------------------

def build_few_shot_prompt(feedback):

    examples_text = ""

    for number, example in enumerate(few_shot_examples, start=1):

        example_output = {
            "category": example["category"],
            "justification": example["justification"],
            "confidence": example["confidence"]
        }

        examples_text += (
            f"\nExample {number}:\n"
            f"Customer feedback: {example['feedback']}\n"
            f"Expected output:\n"
            f"{json.dumps(example_output, indent=2)}\n"
        )

    return f"""
You are a customer feedback classification assistant.

Task:
Classify the following customer feedback into exactly ONE
of the six available categories.

Available categories:
{", ".join(CATEGORIES)}

Category definitions:
{CATEGORY_DEFINITIONS}

Instructions:
1. Read the customer feedback carefully.
2. Identify the customer's main concern.
3. Select the most appropriate category.
4. Provide a short justification based on the feedback.
5. Assign confidence as low, medium, or high.
6. Return only a JSON object with these three fields:
   category, justification, confidence.

Below are completed examples showing the expected format.

{examples_text}

Now classify the following new customer feedback.

Customer feedback:
{feedback}

Return only the JSON object.
"""


# ------------------------------------------------------------
# STEP 3: RUN FEW-SHOT CLASSIFICATION ON ALL 10 TEST ITEMS
# ------------------------------------------------------------

few_shot_records = []

print("=" * 80)
print("PART 3: FEW-SHOT CUSTOMER FEEDBACK CLASSIFICATION")
print(f"Model: {MODEL_NAME}")
print(f"Settings: {MODEL_OPTIONS}")
print(f"Completed examples: {len(few_shot_examples)}")
print("=" * 80)

for index, (feedback, expected) in enumerate(
    zip(test_queries, expected_labels), start=1
):

    print(f"\nProcessing feedback {index}/{len(test_queries)}...")

    try:
        prompt = build_few_shot_prompt(feedback)

        # Reuse the Ollama function from Part 1.
        response = send_prompt(prompt)

        category = response.get("category")
        justification = response.get("justification")
        confidence = response.get("confidence")

        missing_fields = [
            field for field in ("category", "justification", "confidence")
            if field not in response
            or response[field] is None
            or (
                isinstance(response[field], str)
                and not response[field].strip()
            )
        ]

        extra_fields = set(response) - {
            "category", "justification", "confidence"
        }

        formatting_issues = []

        if missing_fields:
            formatting_issues.append(
                "Missing fields: " + ", ".join(missing_fields)
            )

        if category not in CATEGORIES:
            formatting_issues.append("Invalid category")

        if confidence not in CONFIDENCE_LEVELS:
            formatting_issues.append("Invalid confidence")

        if not isinstance(justification, str):
            formatting_issues.append("Justification must be text")

        if extra_fields:
            formatting_issues.append(
                "Unexpected fields: " + ", ".join(sorted(extra_fields))
            )

        valid_output = len(formatting_issues) == 0
        correct = valid_output and category == expected

        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Few-Shot Category": category,
            "Justification": justification,
            "Confidence": confidence,
            "Valid Output": valid_output,
            "Correct": correct,
            "Formatting Issues": (
                "; ".join(formatting_issues)
                if formatting_issues else "None"
            )
        }

    except Exception as error:

        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Few-Shot Category": None,
            "Justification": "",
            "Confidence": None,
            "Valid Output": False,
            "Correct": False,
            "Formatting Issues": str(error)
        }

    few_shot_records.append(record)

    print("Feedback:", feedback)
    print("Expected:", record["Expected"])
    print("Few-Shot Category:", record["Few-Shot Category"])
    print("Justification:", record["Justification"])
    print("Confidence:", record["Confidence"])
    print("Correct:", record["Correct"])
    print("Formatting Issues:", record["Formatting Issues"])
    print("-" * 80)


# ------------------------------------------------------------
# STEP 4: DISPLAY FEW-SHOT RESULTS
# ------------------------------------------------------------

few_shot_df = pd.DataFrame(few_shot_records)

print("\n" + "=" * 80)
print("FEW-SHOT CLASSIFICATION RESULTS")
print("=" * 80)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", 180
):
    display(few_shot_df)


# ------------------------------------------------------------
# STEP 5: CALCULATE FEW-SHOT ACCURACY
# ------------------------------------------------------------

total_items = len(few_shot_df)
correct_items = int(few_shot_df["Correct"].sum())
valid_items = int(few_shot_df["Valid Output"].sum())

few_shot_accuracy = (
    correct_items / total_items * 100
    if total_items else 0
)

print("\n" + "=" * 80)
print("FEW-SHOT EVALUATION SUMMARY")
print("=" * 80)

print(f"Total test items: {total_items}")
print(f"Correct classifications: {correct_items}/{total_items}")
print(f"Incorrect classifications: {total_items - correct_items}")
print(f"Few-shot accuracy: {few_shot_accuracy:.2f}%")
print(f"Valid responses: {valid_items}/{total_items}")
print(f"Invalid responses: {total_items - valid_items}")


# ------------------------------------------------------------
# STEP 6: COMPARE ZERO-SHOT AND FEW-SHOT RESULTS
# ------------------------------------------------------------

# Use the actual zero-shot predictions from Part 2.
zero_shot_comparison = zero_shot_df[
    ["Item", "Predicted"]
].rename(
    columns={"Predicted": "Zero-Shot Category"}
)

few_shot_comparison = few_shot_df[
    ["Item", "Feedback", "Expected", "Few-Shot Category"]
]

comparison_table = few_shot_comparison.merge(
    zero_shot_comparison,
    on="Item",
    how="left",
    validate="one_to_one"
)

comparison_table = comparison_table[
    [
        "Item",
        "Feedback",
        "Expected",
        "Zero-Shot Category",
        "Few-Shot Category"
    ]
]

comparison_table["Zero-Shot Correct"] = (
    comparison_table["Zero-Shot Category"]
    == comparison_table["Expected"]
)

comparison_table["Few-Shot Correct"] = (
    comparison_table["Few-Shot Category"]
    == comparison_table["Expected"]
)

comparison_table["Methods Agree"] = (
    comparison_table["Zero-Shot Category"].notna()
    & comparison_table["Few-Shot Category"].notna()
    & (
        comparison_table["Zero-Shot Category"]
        == comparison_table["Few-Shot Category"]
    )
)

print("\n" + "=" * 80)
print("ZERO-SHOT VS FEW-SHOT COMPARISON")
print("=" * 80)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", 180
):
    display(comparison_table)


# ------------------------------------------------------------
# STEP 7: COMPARE ACCURACY
# ------------------------------------------------------------

zero_shot_correct = int(zero_shot_df["Correct"].sum())
zero_shot_total = len(zero_shot_df)

zero_shot_accuracy = (
    zero_shot_correct / zero_shot_total * 100
    if zero_shot_total else 0
)

accuracy_comparison = pd.DataFrame([
    {
        "Prompting Method": "Zero-Shot",
        "Correct": zero_shot_correct,
        "Total": zero_shot_total,
        "Accuracy (%)": round(zero_shot_accuracy, 2)
    },
    {
        "Prompting Method": "Few-Shot",
        "Correct": correct_items,
        "Total": total_items,
        "Accuracy (%)": round(few_shot_accuracy, 2)
    }
])

print("\n" + "=" * 80)
print("ACCURACY COMPARISON")
print("=" * 80)

display(accuracy_comparison)

agreement_count = int(comparison_table["Methods Agree"].sum())

print(
    f"\nAgreement between methods: "
    f"{agreement_count}/{len(comparison_table)}"
)

print(
    f"Accuracy difference (Few-Shot - Zero-Shot): "
    f"{few_shot_accuracy - zero_shot_accuracy:+.2f} percentage points"
)


# ------------------------------------------------------------
# STEP 8: CHECK INVALID OUTPUTS AND FORMATTING ISSUES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FEW-SHOT FORMATTING ISSUES")
print("=" * 80)

invalid_results = few_shot_df[
    ~few_shot_df["Valid Output"]
]

if invalid_results.empty:
    print("No invalid categories, missing fields, or formatting issues.")
else:
    display(
        invalid_results[
            ["Item", "Feedback", "Formatting Issues"]
        ]
    )


# ------------------------------------------------------------
# STEP 9: SAVE RESULTS
# ------------------------------------------------------------

few_shot_df.to_csv(
    "part3_few_shot_results.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_table.to_csv(
    "part3_zero_shot_vs_few_shot.csv",
    index=False,
    encoding="utf-8-sig"
)

accuracy_comparison.to_csv(
    "part3_accuracy_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print("- part3_few_shot_results.csv")
print("- part3_zero_shot_vs_few_shot.csv")
print("- part3_accuracy_comparison.csv")
print("\nPart 3 execution completed.")

print("""
Q5: Did few-shot prompting improve the results? Explain which outputs changed and whether those changes were improvements.
""")

print("\nAnswer 5")

print("""
Few-shot prompting did not improve the overall accuracy because both zero-shot and few-shot prompting correctly classified all 10 feedback items, 
giving both methods 100% accuracy. None of the predicted categories changed between the two methods. However, I noticed a difference in the 
confidence ratings. With zero-shot prompting, the nonprofit discount question received low confidence, and the biometric login request received 
medium confidence. With few-shot prompting, both received high confidence. The justifications also changed slightly but still explained the 
classifications correctly. Overall, few-shot prompting produced more consistent confidence ratings, but it did not improve classification accuracy. 
Higher confidence does not necessarily mean the model became more accurate.
""")

print("""
Q6: How might the examples you selected have influenced the model? Identify one way that poorly chosen, unrepresentative, or biased examples 
could reduce performance.

""")

print("\nAnswer 6")

print("""
The six examples selected may have helped the model understand how to classify customer feedback and what the expected responses should look like. 
I included one example for each category, such as duplicate invoices for BILLING, application crashes for TECHNICAL_ISSUE, and dark mode requests 
for FEATURE_REQUEST. These examples showed the model how to connect customer messages to the correct categories while providing short justifications
and confidence ratings. However, poorly chosen examples could reduce performance if they only represent simple situations or focus too much on one 
category. For example, if most examples involved billing problems, the model might incorrectly classify unrelated feedback as BILLING. To avoid this,
I would use balanced examples that represent different categories and include more complicated customer messages.
""")
print("\n***********************End of Few-Shot Prompting*************************")




***********************Part 3: Few-Shot Prompting*************************


PART 3: FEW-SHOT CUSTOMER FEEDBACK CLASSIFICATION
Model: llama3.2
Settings: {'temperature': 0, 'seed': 42, 'num_predict': 220}
Completed examples: 6

Processing feedback 1/10...
Feedback: I was charged twice for my annual subscription and need one of the charges refunded.
Expected: BILLING
Few-Shot Category: BILLING
Justification: The customer reports duplicate charging and requests a refund.
Confidence: high
Correct: True
Formatting Issues: None
--------------------------------------------------------------------------------

Processing feedback 2/10...
Feedback: The dashboard freezes whenever I try to export a report as a PDF.
Expected: TECHNICAL_ISSUE
Few-Shot Category: TECHNICAL_ISSUE
Justification: The customer reports an application freeze when exporting a report.
Confidence: high
Correct: True
Formatting Issues: None
--------------------------------------------------------------------------------

Pro

,Item,Feedback,Expected,Few-Shot Category,Justification,Confidence,Valid Output,Correct,Formatting Issues
0,1,I was charged twice for my annual subscription and need one of the charges refunded.,BILLING,BILLING,The customer reports duplicate charging and requests a refund.,high,True,True,None
1,2,The dashboard freezes whenever I try to export a report as a PDF.,TECHNICAL_ISSUE,TECHNICAL_ISSUE,The customer reports an application freeze when exporting a report.,high,True,True,None
2,3,It would be helpful if we could schedule reports to be emailed automatically.,FEATURE_REQUEST,FEATURE_REQUEST,The customer requests a feature to automate report emailing.,high,True,True,None
3,4,I have contacted support three times and am thinking about switching to another platform.,CANCELLATION_RISK,CANCELLATION_RISK,The customer mentions repeated contact with support and consideration of switching providers.,high,True,True,None
4,5,The new dashboard is much easier to use. Great update!,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,The customer praises the new dashboard.,high,True,True,None
5,6,Can someone explain whether you offer discounts for nonprofit organizations?,BILLING,BILLING,The customer asks about potential discounts for nonprofit organizations.,high,True,True,None
6,7,My payment failed after my company replaced its corporate credit card.,BILLING,BILLING,The customer reports a failed payment due to a change in their corporate credit card.,high,True,True,None
7,8,"The mobile app works, but I wish it supported biometric login.",FEATURE_REQUEST,FEATURE_REQUEST,The customer requests a new feature (biometric login) for the mobile app.,high,True,True,None
8,9,"Everything has been slow this week, and our team cannot complete its work.",TECHNICAL_ISSUE,TECHNICAL_ISSUE,The customer reports a performance problem with slow speeds.,high,True,True,None
9,10,Thanks to Jordan from support for resolving my issue so quickly.,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,The customer praises the support team's efficiency.,high,True,True,None



FEW-SHOT EVALUATION SUMMARY
Total test items: 10
Correct classifications: 10/10
Incorrect classifications: 0
Few-shot accuracy: 100.00%
Valid responses: 10/10
Invalid responses: 0

ZERO-SHOT VS FEW-SHOT COMPARISON


,Item,Feedback,Expected,Zero-Shot Category,Few-Shot Category,Zero-Shot Correct,Few-Shot Correct,Methods Agree
0,1,I was charged twice for my annual subscription and need one of the charges refunded.,BILLING,BILLING,BILLING,True,True,True
1,2,The dashboard freezes whenever I try to export a report as a PDF.,TECHNICAL_ISSUE,TECHNICAL_ISSUE,TECHNICAL_ISSUE,True,True,True
2,3,It would be helpful if we could schedule reports to be emailed automatically.,FEATURE_REQUEST,FEATURE_REQUEST,FEATURE_REQUEST,True,True,True
3,4,I have contacted support three times and am thinking about switching to another platform.,CANCELLATION_RISK,CANCELLATION_RISK,CANCELLATION_RISK,True,True,True
4,5,The new dashboard is much easier to use. Great update!,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,True,True,True
5,6,Can someone explain whether you offer discounts for nonprofit organizations?,BILLING,BILLING,BILLING,True,True,True
6,7,My payment failed after my company replaced its corporate credit card.,BILLING,BILLING,BILLING,True,True,True
7,8,"The mobile app works, but I wish it supported biometric login.",FEATURE_REQUEST,FEATURE_REQUEST,FEATURE_REQUEST,True,True,True
8,9,"Everything has been slow this week, and our team cannot complete its work.",TECHNICAL_ISSUE,TECHNICAL_ISSUE,TECHNICAL_ISSUE,True,True,True
9,10,Thanks to Jordan from support for resolving my issue so quickly.,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,True,True,True



ACCURACY COMPARISON


,Prompting Method,Correct,Total,Accuracy (%)
0,Zero-Shot,10,10,100.0
1,Few-Shot,10,10,100.0



Agreement between methods: 10/10
Accuracy difference (Few-Shot - Zero-Shot): +0.00 percentage points

FEW-SHOT FORMATTING ISSUES
No invalid categories, missing fields, or formatting issues.

Saved:
- part3_few_shot_results.csv
- part3_zero_shot_vs_few_shot.csv
- part3_accuracy_comparison.csv

Part 3 execution completed.

Q5: Did few-shot prompting improve the results? Explain which outputs changed and whether those changes were improvements.


Answer 5

Few-shot prompting did not improve the overall accuracy because both zero-shot and few-shot prompting correctly classified all 10 feedback items, 
giving both methods 100% accuracy. None of the predicted categories changed between the two methods. However, I noticed a difference in the 
confidence ratings. With zero-shot prompting, the nonprofit discount question received low confidence, and the biometric login request received 
medium confidence. With few-shot prompting, both received high confidence. The justifications also changed s

# Part 4: Reasoning-Oriented Prompting

Reasoning-oriented prompts ask the model to consider relevant distinctions or apply a decision process before selecting an answer. The goal is not to collect a lengthy internal chain of thought, but to encourage deliberate classification and obtain a concise, useful justification.

**TODO:**

* Develop a reasoning-oriented version of your prompt.
* Run this prompt on the same ten test items.
* Rebuild your comparison table from above with the Reasoning-Oriented results.
* Calculate its classification accuracy.
* Record any cases in which the justification revealed a weak or inconsistent classification.

You can use any of the reasoning-based methods discussed in this week's lecture, media, or readings, but you might consider asking for justification, defining a decision sequence, and/or instructing the model to consider something specific before answering. You do not need to include an extended hidden reasoning trace unless you want to do so.

**Q7:** Did the reasoning-oriented prompt improve classification quality, justification quality, or both? Support your answer with specific results.

**A7:**

**Q8:** Did the model ever provide a plausible justification for a questionable or incorrect answer? What does this suggest about using model explanations as evidence of correctness?

**A8:**

In [30]:
print("\n")
print("***********************Part 4: Reasoning-Oriented Prompting*************************")
print("\n")

# ============================================================
# PART 4: REASONING-ORIENTED PROMPTING
# AAI-530: Prompt-Based Learning Comparison
# ============================================================

# Reuse existing variables and functions from Parts 1-3:
# MODEL_NAME, MODEL_OPTIONS, CATEGORIES, CONFIDENCE_LEVELS
# CATEGORY_DEFINITIONS, test_queries, expected_labels
# send_prompt(), zero_shot_df, few_shot_df, pd, display

# ------------------------------------------------------------
# STEP 1: CREATE A REASONING-ORIENTED PROMPT
# ------------------------------------------------------------

def build_reasoning_prompt(feedback):
    return f"""
You are a customer feedback classification assistant.

Task:
Classify customer feedback into exactly ONE of the
six available categories.

Available categories:
{", ".join(CATEGORIES)}

Category definitions:
{CATEGORY_DEFINITIONS}

Decision process:
Before choosing a category, carefully consider:

1. What is the customer's main concern or request?

2. Does the customer explicitly mention canceling,
   switching providers, or leaving the service?
   If yes, prioritize CANCELLATION_RISK.

3. Is the customer discussing charges, refunds,
   payments, invoices, pricing, or discounts?
   If yes, consider BILLING.

4. Is the customer describing an existing feature
   that is broken, freezing, crashing, or running slowly?
   If yes, consider TECHNICAL_ISSUE.

5. Is the customer asking for a new feature,
   integration, or improvement?
   If yes, consider FEATURE_REQUEST.

6. Is the customer praising the product,
   service, or support?
   If yes, consider POSITIVE_FEEDBACK.

7. If none of these categories clearly apply,
   select OTHER.

8. If more than one category seems possible,
   choose the category that best represents the
   customer's main concern. Do not assume the
   customer wants to cancel unless they indicate it.

9. Assign a confidence rating:
   - high: The category is clearly supported.
   - medium: More than one category is reasonable.
   - low: The feedback is unclear or lacks detail.

Output instructions:
- Return exactly one category.
- Provide a short, evidence-based justification.
- Include a confidence rating: low, medium, or high.
- Do not provide extended reasoning.
- Return only a valid JSON object containing:
  category, justification, confidence.

Customer feedback:
{feedback}
"""


# ------------------------------------------------------------
# STEP 2: RUN REASONING-ORIENTED PROMPT ON ALL 10 ITEMS
# ------------------------------------------------------------

reasoning_records = []

print("=" * 80)
print("PART 4: REASONING-ORIENTED CLASSIFICATION")
print(f"Model: {MODEL_NAME}")
print(f"Settings: {MODEL_OPTIONS}")
print("=" * 80)

for index, (feedback, expected) in enumerate(
    zip(test_queries, expected_labels), start=1
):

    print(f"\nProcessing feedback {index}/{len(test_queries)}...")

    try:
        prompt = build_reasoning_prompt(feedback)

        # Reuse the model connection from Part 1.
        response = send_prompt(prompt)

        category = response.get("category")
        justification = response.get("justification")
        confidence = response.get("confidence")

        required_fields = (
            "category", "justification", "confidence"
        )

        missing_fields = [
            field for field in required_fields
            if field not in response
            or response[field] is None
            or (
                isinstance(response[field], str)
                and not response[field].strip()
            )
        ]

        extra_fields = set(response) - set(required_fields)

        formatting_issues = []

        if missing_fields:
            formatting_issues.append(
                "Missing fields: " + ", ".join(missing_fields)
            )

        if category not in CATEGORIES:
            formatting_issues.append("Invalid category")

        if confidence not in CONFIDENCE_LEVELS:
            formatting_issues.append("Invalid confidence")

        if not isinstance(justification, str):
            formatting_issues.append(
                "Justification must be text"
            )

        if extra_fields:
            formatting_issues.append(
                "Unexpected fields: " +
                ", ".join(sorted(extra_fields))
            )

        valid_output = len(formatting_issues) == 0
        correct = valid_output and category == expected

        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Reasoning-Oriented Category": category,
            "Justification": justification,
            "Confidence": confidence,
            "Valid Output": valid_output,
            "Correct": correct,
            "Formatting Issues": (
                "; ".join(formatting_issues)
                if formatting_issues else "None"
            )
        }

    except Exception as error:

        record = {
            "Item": index,
            "Feedback": feedback,
            "Expected": expected,
            "Reasoning-Oriented Category": None,
            "Justification": "",
            "Confidence": None,
            "Valid Output": False,
            "Correct": False,
            "Formatting Issues": str(error)
        }

    reasoning_records.append(record)

    print("Feedback:", feedback)
    print("Expected:", record["Expected"])
    print(
        "Reasoning-Oriented Category:",
        record["Reasoning-Oriented Category"]
    )
    print("Justification:", record["Justification"])
    print("Confidence:", record["Confidence"])
    print("Correct:", record["Correct"])
    print("Formatting Issues:", record["Formatting Issues"])
    print("-" * 80)


# ------------------------------------------------------------
# STEP 3: DISPLAY REASONING-ORIENTED RESULTS
# ------------------------------------------------------------

reasoning_df = pd.DataFrame(reasoning_records)

print("\n" + "=" * 80)
print("REASONING-ORIENTED CLASSIFICATION RESULTS")
print("=" * 80)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", 180
):
    display(reasoning_df)


# ------------------------------------------------------------
# STEP 4: CALCULATE REASONING-ORIENTED ACCURACY
# ------------------------------------------------------------

total_items = len(reasoning_df)
correct_items = int(reasoning_df["Correct"].sum())
valid_items = int(reasoning_df["Valid Output"].sum())

reasoning_accuracy = (
    correct_items / total_items * 100
    if total_items else 0
)

print("\n" + "=" * 80)
print("REASONING-ORIENTED EVALUATION SUMMARY")
print("=" * 80)

print(f"Total test items: {total_items}")
print(f"Correct classifications: {correct_items}/{total_items}")
print(f"Incorrect classifications: {total_items - correct_items}")
print(f"Reasoning-oriented accuracy: {reasoning_accuracy:.2f}%")
print(f"Valid responses: {valid_items}/{total_items}")
print(f"Invalid responses: {total_items - valid_items}")


# ------------------------------------------------------------
# STEP 5: REBUILD THE THREE-METHOD COMPARISON TABLE
# ------------------------------------------------------------

zero_comparison = zero_shot_df[
    ["Item", "Predicted"]
].rename(
    columns={"Predicted": "Zero-Shot Category"}
)

few_comparison = few_shot_df[
    ["Item", "Few-Shot Category"]
]

reasoning_comparison = reasoning_df[
    ["Item", "Feedback", "Expected",
     "Reasoning-Oriented Category"]
]

three_method_comparison = (
    reasoning_comparison
    .merge(
        zero_comparison,
        on="Item",
        how="left",
        validate="one_to_one"
    )
    .merge(
        few_comparison,
        on="Item",
        how="left",
        validate="one_to_one"
    )
)

three_method_comparison = three_method_comparison[
    [
        "Item",
        "Feedback",
        "Expected",
        "Zero-Shot Category",
        "Few-Shot Category",
        "Reasoning-Oriented Category"
    ]
]

for method_column in [
    "Zero-Shot Category",
    "Few-Shot Category",
    "Reasoning-Oriented Category"
]:
    three_method_comparison[
        method_column.replace(" Category", " Correct")
    ] = (
        three_method_comparison[method_column]
        == three_method_comparison["Expected"]
    )

method_columns = [
    "Zero-Shot Category",
    "Few-Shot Category",
    "Reasoning-Oriented Category"
]

three_method_comparison["All Methods Agree"] = (
    three_method_comparison[method_columns].notna().all(axis=1)
    & (
        three_method_comparison[method_columns]
        .nunique(axis=1) == 1
    )
)

print("\n" + "=" * 80)
print("ZERO-SHOT VS FEW-SHOT VS REASONING-ORIENTED")
print("=" * 80)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", 180
):
    display(three_method_comparison)


# ------------------------------------------------------------
# STEP 6: COMPARE ACCURACY ACROSS ALL THREE METHODS
# ------------------------------------------------------------

zero_shot_accuracy = (
    zero_shot_df["Correct"].sum()
    / len(zero_shot_df) * 100
)

few_shot_accuracy = (
    few_shot_df["Correct"].sum()
    / len(few_shot_df) * 100
)

accuracy_comparison_all = pd.DataFrame([
    {
        "Prompting Method": "Zero-Shot",
        "Correct": int(zero_shot_df["Correct"].sum()),
        "Total": len(zero_shot_df),
        "Accuracy (%)": round(zero_shot_accuracy, 2)
    },
    {
        "Prompting Method": "Few-Shot",
        "Correct": int(few_shot_df["Correct"].sum()),
        "Total": len(few_shot_df),
        "Accuracy (%)": round(few_shot_accuracy, 2)
    },
    {
        "Prompting Method": "Reasoning-Oriented",
        "Correct": correct_items,
        "Total": total_items,
        "Accuracy (%)": round(reasoning_accuracy, 2)
    }
])

print("\n" + "=" * 80)
print("THREE-METHOD ACCURACY COMPARISON")
print("=" * 80)

display(accuracy_comparison_all)

agreement_count = int(
    three_method_comparison["All Methods Agree"].sum()
)

print(
    f"\nAgreement across all three methods: "
    f"{agreement_count}/{len(three_method_comparison)}"
)


# ------------------------------------------------------------
# STEP 7: IDENTIFY WEAK OR INCONSISTENT JUSTIFICATIONS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("JUSTIFICATION QUALITY REVIEW")
print("=" * 80)

# Flag responses needing human review.
# A low confidence rating, incorrect prediction, or invalid
# output is a review flag, not proof of weak reasoning.

reasoning_df["Needs Review"] = (
    ~reasoning_df["Correct"]
    | ~reasoning_df["Valid Output"]
    | reasoning_df["Confidence"].eq("low")
    | reasoning_df["Justification"].fillna("").str.strip().eq("")
)

review_items = reasoning_df[
    reasoning_df["Needs Review"]
]

if review_items.empty:
    print("No automatic review flags detected.")
else:
    display(
        review_items[
            [
                "Item",
                "Feedback",
                "Expected",
                "Reasoning-Oriented Category",
                "Justification",
                "Confidence",
                "Formatting Issues"
            ]
        ]
    )

print(
    "\nManually review justifications for unsupported claims, "
    "incorrect priorities, and inconsistent category choices."
)

# Human review notes can be recorded here after inspection.
reasoning_df["Human Review Notes"] = ""

# Example: after reviewing item 4, enter a note if needed:
# reasoning_df.loc[
#     reasoning_df["Item"] == 4, "Human Review Notes"
# ] = "Correctly prioritized explicit switching risk."


# ------------------------------------------------------------
# STEP 8: CHECK FORMATTING PROBLEMS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FORMATTING ISSUES")
print("=" * 80)

invalid_results = reasoning_df[
    ~reasoning_df["Valid Output"]
]

if invalid_results.empty:
    print(
        "No invalid categories, missing fields, "
        "or formatting issues detected."
    )
else:
    display(
        invalid_results[
            ["Item", "Feedback", "Formatting Issues"]
        ]
    )


# ------------------------------------------------------------
# STEP 9: SAVE ALL RESULTS
# ------------------------------------------------------------

reasoning_df.to_csv(
    "part4_reasoning_oriented_results.csv",
    index=False,
    encoding="utf-8-sig"
)

three_method_comparison.to_csv(
    "part4_three_method_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

accuracy_comparison_all.to_csv(
    "part4_accuracy_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print("- part4_reasoning_oriented_results.csv")
print("- part4_three_method_comparison.csv")
print("- part4_accuracy_comparison.csv")
print("\nPart 4 execution completed.")

print("""
Q7: Did the reasoning-oriented prompt improve classification quality, justification quality, or both? Support your answer with specific results.
""")

print("\nAnswer 7")

print("""
The reasoning-oriented prompt did not improve classification accuracy because all three methods correctly classified all 10 feedback items, 
giving each method 100% accuracy. However, it helped produce clear justifications that directly supported the selected categories. For example, 
the model correctly classified the customer thinking about switching platforms as CANCELLATION_RISK and explained that the customer was considering 
another platform. It also identified the nonprofit discount question as BILLING because it involved pricing. All 10 responses had high confidence, 
compared to the zero-shot results, which included one low and one medium confidence rating. There were no formatting errors or incorrect 
classifications. Overall, the reasoning-oriented prompt produced consistent explanations and confidence ratings, but it did not demonstrate a 
clear improvement in accuracy or justification quality over the other methods.
""")

print("""
Q8: Did the model ever provide a plausible justification for a questionable or incorrect answer? What does this suggest about using model explanations 
as evidence of correctness?
""")

print("\nAnswer 8")

print("""
In this experiment, the model did not provide any incorrect classifications. All 10 reasoning-oriented predictions matched the expected categories, 
and the justifications were consistent with the customer feedback. However, some messages could reasonably fit more than one category. For example, 
the customer who contacted support three times and considered switching platforms could be seen as a support complaint, but the model correctly 
focused on CANCELLATION_RISK. This shows that even when a model provides a reasonable explanation, it does not automatically mean the classification 
is correct. A model can sound convincing while overlooking important details or choosing the wrong category. Because of this, I would still compare 
its predictions with expected labels and review the explanations instead of relying on the justifications alone.
""")



print("\n***********************End of Reasoning-Oriented Prompting*************************")



***********************Part 4: Reasoning-Oriented Prompting*************************


PART 4: REASONING-ORIENTED CLASSIFICATION
Model: llama3.2
Settings: {'temperature': 0, 'seed': 42, 'num_predict': 220}

Processing feedback 1/10...
Feedback: I was charged twice for my annual subscription and need one of the charges refunded.
Expected: BILLING
Reasoning-Oriented Category: BILLING
Justification: Customer mentions being charged twice for their annual subscription and requests a refund.
Confidence: high
Correct: True
Formatting Issues: None
--------------------------------------------------------------------------------

Processing feedback 2/10...
Feedback: The dashboard freezes whenever I try to export a report as a PDF.
Expected: TECHNICAL_ISSUE
Reasoning-Oriented Category: TECHNICAL_ISSUE
Justification: The customer is describing a technical problem with the dashboard freezing when exporting a report as a PDF.
Confidence: high
Correct: True
Formatting Issues: None
----------------

,Item,Feedback,Expected,Reasoning-Oriented Category,Justification,Confidence,Valid Output,Correct,Formatting Issues
0,1,I was charged twice for my annual subscription and need one of the charges refunded.,BILLING,BILLING,Customer mentions being charged twice for their annual subscription and requests a refund.,high,True,True,None
1,2,The dashboard freezes whenever I try to export a report as a PDF.,TECHNICAL_ISSUE,TECHNICAL_ISSUE,The customer is describing a technical problem with the dashboard freezing when exporting a report as a PDF.,high,True,True,None
2,3,It would be helpful if we could schedule reports to be emailed automatically.,FEATURE_REQUEST,FEATURE_REQUEST,Customer requests scheduling reports to be emailed automatically.,high,True,True,None
3,4,I have contacted support three times and am thinking about switching to another platform.,CANCELLATION_RISK,CANCELLATION_RISK,Customer mentions considering switching to another platform.,high,True,True,None
4,5,The new dashboard is much easier to use. Great update!,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,"Customer praises the new dashboard, indicating satisfaction with the product update.",high,True,True,None
5,6,Can someone explain whether you offer discounts for nonprofit organizations?,BILLING,BILLING,"The customer is asking about discounts for nonprofit organizations, which is related to pricing and billing.",high,True,True,None
6,7,My payment failed after my company replaced its corporate credit card.,BILLING,BILLING,Customer mentions a payment failure after a corporate credit card replacement.,high,True,True,None
7,8,"The mobile app works, but I wish it supported biometric login.",FEATURE_REQUEST,FEATURE_REQUEST,The customer is requesting a new feature (biometric login) for the mobile app.,high,True,True,None
8,9,"Everything has been slow this week, and our team cannot complete its work.",TECHNICAL_ISSUE,TECHNICAL_ISSUE,"Customer mentions that everything has been slow, indicating a performance problem.",high,True,True,None
9,10,Thanks to Jordan from support for resolving my issue so quickly.,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,Customer praises the support team for resolving their issue quickly.,high,True,True,None



REASONING-ORIENTED EVALUATION SUMMARY
Total test items: 10
Correct classifications: 10/10
Incorrect classifications: 0
Reasoning-oriented accuracy: 100.00%
Valid responses: 10/10
Invalid responses: 0

ZERO-SHOT VS FEW-SHOT VS REASONING-ORIENTED


,Item,Feedback,Expected,Zero-Shot Category,Few-Shot Category,Reasoning-Oriented Category,Zero-Shot Correct,Few-Shot Correct,Reasoning-Oriented Correct,All Methods Agree
0,1,I was charged twice for my annual subscription and need one of the charges refunded.,BILLING,BILLING,BILLING,BILLING,True,True,True,True
1,2,The dashboard freezes whenever I try to export a report as a PDF.,TECHNICAL_ISSUE,TECHNICAL_ISSUE,TECHNICAL_ISSUE,TECHNICAL_ISSUE,True,True,True,True
2,3,It would be helpful if we could schedule reports to be emailed automatically.,FEATURE_REQUEST,FEATURE_REQUEST,FEATURE_REQUEST,FEATURE_REQUEST,True,True,True,True
3,4,I have contacted support three times and am thinking about switching to another platform.,CANCELLATION_RISK,CANCELLATION_RISK,CANCELLATION_RISK,CANCELLATION_RISK,True,True,True,True
4,5,The new dashboard is much easier to use. Great update!,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,True,True,True,True
5,6,Can someone explain whether you offer discounts for nonprofit organizations?,BILLING,BILLING,BILLING,BILLING,True,True,True,True
6,7,My payment failed after my company replaced its corporate credit card.,BILLING,BILLING,BILLING,BILLING,True,True,True,True
7,8,"The mobile app works, but I wish it supported biometric login.",FEATURE_REQUEST,FEATURE_REQUEST,FEATURE_REQUEST,FEATURE_REQUEST,True,True,True,True
8,9,"Everything has been slow this week, and our team cannot complete its work.",TECHNICAL_ISSUE,TECHNICAL_ISSUE,TECHNICAL_ISSUE,TECHNICAL_ISSUE,True,True,True,True
9,10,Thanks to Jordan from support for resolving my issue so quickly.,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,True,True,True,True



THREE-METHOD ACCURACY COMPARISON


,Prompting Method,Correct,Total,Accuracy (%)
0,Zero-Shot,10,10,100.0
1,Few-Shot,10,10,100.0
2,Reasoning-Oriented,10,10,100.0



Agreement across all three methods: 10/10

JUSTIFICATION QUALITY REVIEW
No automatic review flags detected.

Manually review justifications for unsupported claims, incorrect priorities, and inconsistent category choices.

FORMATTING ISSUES
No invalid categories, missing fields, or formatting issues detected.

Saved:
- part4_reasoning_oriented_results.csv
- part4_three_method_comparison.csv
- part4_accuracy_comparison.csv

Part 4 execution completed.

Q7: Did the reasoning-oriented prompt improve classification quality, justification quality, or both? Support your answer with specific results.


Answer 7

The reasoning-oriented prompt did not improve classification accuracy because all three methods correctly classified all 10 feedback items, 
giving each method 100% accuracy. However, it helped produce clear justifications that directly supported the selected categories. For example, 
the model correctly classified the customer thinking about switching platforms as CANCELLATION_RISK a

# Part 5: Reusable Prompt Template and Final Evaluation

A prompt template separates stable instructions from changing user input so that the same workflow can be reused across many examples or applications.

**TODO:**

* Create a final reusable prompt template based on what you learned from Parts 2–4.
* Implement the template as a reusable function, formatted string, or equivalent workflow.
* Test the final template on at least three new feedback items that you write yourself., including at least one intentionally ambiguous or difficult item.
* Display the final results clearly.

Your template should include separate items for:

  * a system instruction or model role
  * task instructions
  * category definitions
  * optional few-shot examples
  * the changing customer-feedback input
  * output-format requirements

**Q9:** Which prompting method or combination of methods would you use in the final application? Explain why it was the best fit for this task.

**A9:**

**Q10:** Describe the intentionally difficult test item you created. How did the model handle it, and what does that result reveal about the limits of your prompt template?

**A10:**

**Q11:** What changes would be needed to adapt your template to a different task, such as summarization, content generation, or question answering?

**A11:**

In [31]:

print("\n")
print("***********************Part 5: Reusable Prompt Template and Final Evaluation*************************")
print("\n")

# ============================================================
# PART 5: REUSABLE PROMPT TEMPLATE AND FINAL EVALUATION
# AAI-530: Prompt-Based Learning Comparison
# ============================================================

# Reuse existing imports, variables, and functions from
# Parts 1-4. Do not repeat imports or model setup.

# Existing:
# pd, json, ollama, display
# MODEL_NAME, MODEL_OPTIONS, RESPONSE_SCHEMA
# CATEGORIES, CATEGORY_DEFINITIONS, CONFIDENCE_LEVELS
# few_shot_examples


# ------------------------------------------------------------
# STEP 1: DEFINE THE REUSABLE SYSTEM INSTRUCTION
# ------------------------------------------------------------

FINAL_SYSTEM_ROLE = """
You are a customer feedback classification assistant.

Your responsibility is to accurately classify customer
feedback into one of six predefined categories.

Use only the information provided in the customer message.

Choose the category that best represents the customer's
main concern.

Provide a short, evidence-based justification.

Do not invent details or provide extended reasoning.
"""


# ------------------------------------------------------------
# STEP 2: DEFINE STABLE TASK INSTRUCTIONS
# ------------------------------------------------------------

FINAL_TASK_INSTRUCTIONS = """
Task:
Classify each customer feedback message into exactly
one of the available categories.

Decision guidelines:

1. Identify the customer's main concern.

2. If the customer explicitly mentions canceling,
   switching providers, or leaving the service,
   consider CANCELLATION_RISK.

3. If the message discusses payments, charges,
   refunds, invoices, pricing, or discounts,
   consider BILLING.

4. If the customer describes something broken,
   crashing, freezing, failing, or running slowly,
   consider TECHNICAL_ISSUE.

5. If the customer requests a new feature,
   integration, or improvement,
   consider FEATURE_REQUEST.

6. If the customer expresses satisfaction or praise,
   consider POSITIVE_FEEDBACK.

7. If no other category applies, select OTHER.

8. If multiple categories seem possible,
   select the category that represents the main
   concern or most important customer intent.

9. Do not assume cancellation risk unless the
   customer indicates an intention to leave.

Confidence guidelines:

- high: The classification is clearly supported.
- medium: Multiple categories are reasonable.
- low: The feedback is unclear or lacks detail.
"""


# ------------------------------------------------------------
# STEP 3: DEFINE OUTPUT FORMAT REQUIREMENTS
# ------------------------------------------------------------

FINAL_OUTPUT_REQUIREMENTS = """
Output requirements:

Return only a valid JSON object containing exactly
these three fields:

{
  "category": "ONE_VALID_CATEGORY",
  "justification": "Short explanation based on the feedback",
  "confidence": "low, medium, or high"
}

Rules:
- Select exactly one category.
- Use only the six allowed category labels.
- Keep the justification concise and specific.
- Do not include extra fields.
- Do not include Markdown or additional text.
"""


# ------------------------------------------------------------
# STEP 4: CREATE THE REUSABLE PROMPT TEMPLATE
# ------------------------------------------------------------

def build_final_prompt(feedback, use_examples=False):
    """
    Build a reusable classification prompt.

    Parameters:
        feedback (str):
            Customer feedback to classify.

        use_examples (bool):
            Whether to include completed few-shot examples.

    Returns:
        str: Fully constructed classification prompt.
    """

    if not isinstance(feedback, str) or not feedback.strip():
        raise ValueError("Feedback must be a nonempty string.")

    # Stable category definitions from Part 1.
    category_section = (
        "Available categories:\n"
        + ", ".join(CATEGORIES)
        + "\n\nCategory definitions:\n"
        + str(CATEGORY_DEFINITIONS)
    )

    # Optional completed examples from Part 3.
    examples_section = ""

    if use_examples:
        examples_section = "\nCompleted examples:\n"

        for number, example in enumerate(
            few_shot_examples, start=1
        ):

            example_output = {
                "category": example["category"],
                "justification": example["justification"],
                "confidence": example["confidence"]
            }

            examples_section += (
                f"\nExample {number}:\n"
                f"Feedback: {example['feedback']}\n"
                f"Output:\n"
                f"{json.dumps(example_output, indent=2)}\n"
            )

    # Changing customer input.
    feedback_section = (
        "\nCustomer feedback to classify:\n"
        + feedback.strip()
    )

    # Combine the reusable sections.
    final_prompt = (
        FINAL_SYSTEM_ROLE
        + "\n"
        + FINAL_TASK_INSTRUCTIONS
        + "\n"
        + category_section
        + "\n"
        + examples_section
        + "\n"
        + FINAL_OUTPUT_REQUIREMENTS
        + "\n"
        + feedback_section
    )

    return final_prompt


# ------------------------------------------------------------
# STEP 5: CREATE A REUSABLE CLASSIFICATION FUNCTION
# ------------------------------------------------------------

def classify_customer_feedback(feedback, use_examples=False):
    """
    Classify one customer feedback message using
    the final reusable prompt template.
    """

    prompt = build_final_prompt(
        feedback=feedback,
        use_examples=use_examples
    )

    # Reuse the Ollama function from Part 1.
    response = send_prompt(prompt)

    required_fields = {
        "category",
        "justification",
        "confidence"
    }

    missing_fields = required_fields - set(response)
    extra_fields = set(response) - required_fields

    issues = []

    if missing_fields:
        issues.append(
            "Missing fields: " +
            ", ".join(sorted(missing_fields))
        )

    if extra_fields:
        issues.append(
            "Unexpected fields: " +
            ", ".join(sorted(extra_fields))
        )

    if response.get("category") not in CATEGORIES:
        issues.append("Invalid category")

    if response.get("confidence") not in CONFIDENCE_LEVELS:
        issues.append("Invalid confidence")

    justification = response.get("justification")

    if (
        not isinstance(justification, str)
        or not justification.strip()
    ):
        issues.append("Missing or invalid justification")

    return {
        "category": response.get("category"),
        "justification": response.get("justification"),
        "confidence": response.get("confidence"),
        "valid_output": len(issues) == 0,
        "formatting_issues": (
            "; ".join(issues) if issues else "None"
        )
    }


# ------------------------------------------------------------
# STEP 6: CREATE FIVE NEW TEST ITEMS
# ------------------------------------------------------------

# These examples are new and were not used in Parts 1-4.
# Items 4 and 5 are intentionally ambiguous or difficult.

final_test_items = [
    {
        "feedback": (
            "I updated my billing address, but my latest "
            "invoice still shows the old address."
        ),
        "expected": "BILLING",
        "difficulty": "Standard"
    },
    {
        "feedback": (
            "The search feature returns an error whenever "
            "I enter a customer's name."
        ),
        "expected": "TECHNICAL_ISSUE",
        "difficulty": "Standard"
    },
    {
        "feedback": (
            "I really appreciate how easy the new "
            "onboarding process has become."
        ),
        "expected": "POSITIVE_FEEDBACK",
        "difficulty": "Standard"
    },
    {
        "feedback": (
            "The reporting tool keeps crashing, and if "
            "this continues, we may cancel our subscription."
        ),
        "expected": "CANCELLATION_RISK",
        "difficulty": "Ambiguous"
    },
    {
        "feedback": (
            "The mobile app is useful, but it would be "
            "better if notifications could be customized."
        ),
        "expected": "FEATURE_REQUEST",
        "difficulty": "Ambiguous"
    }
]


# Verify new test items are not reused from earlier parts.

previous_feedback = set(test_queries)

previous_feedback.update(
    example["feedback"] for example in few_shot_examples
)

assert all(
    item["feedback"] not in previous_feedback
    for item in final_test_items
)

assert len(final_test_items) >= 3

assert any(
    item["difficulty"] == "Ambiguous"
    for item in final_test_items
)


# ------------------------------------------------------------
# STEP 7: RUN FINAL TEMPLATE ON NEW TEST ITEMS
# ------------------------------------------------------------

# Set to True to include the optional Part 3 examples.
# False uses the final decision-oriented template alone.
USE_FEW_SHOT_EXAMPLES = False

final_records = []

print("=" * 80)
print("PART 5: FINAL REUSABLE PROMPT TEMPLATE")
print(f"Model: {MODEL_NAME}")
print(f"Settings: {MODEL_OPTIONS}")
print(f"Optional examples enabled: {USE_FEW_SHOT_EXAMPLES}")
print(f"New test items: {len(final_test_items)}")
print("=" * 80)

for index, item in enumerate(final_test_items, start=1):

    feedback = item["feedback"]
    expected = item["expected"]

    print(
        f"\nProcessing new feedback "
        f"{index}/{len(final_test_items)}..."
    )

    try:
        result = classify_customer_feedback(
            feedback,
            use_examples=USE_FEW_SHOT_EXAMPLES
        )

        predicted = result["category"]

        correct = (
            result["valid_output"]
            and predicted == expected
        )

        record = {
            "Item": index,
            "Feedback": feedback,
            "Difficulty": item["difficulty"],
            "Expected Category": expected,
            "Predicted Category": predicted,
            "Justification": result["justification"],
            "Confidence": result["confidence"],
            "Correct": correct,
            "Valid Output": result["valid_output"],
            "Formatting Issues": result["formatting_issues"]
        }

    except Exception as error:

        record = {
            "Item": index,
            "Feedback": feedback,
            "Difficulty": item["difficulty"],
            "Expected Category": expected,
            "Predicted Category": None,
            "Justification": "",
            "Confidence": None,
            "Correct": False,
            "Valid Output": False,
            "Formatting Issues": str(error)
        }

    final_records.append(record)

    print("Feedback:", record["Feedback"])
    print("Difficulty:", record["Difficulty"])
    print("Expected:", record["Expected Category"])
    print("Predicted:", record["Predicted Category"])
    print("Justification:", record["Justification"])
    print("Confidence:", record["Confidence"])
    print("Correct:", record["Correct"])
    print("-" * 80)


# ------------------------------------------------------------
# STEP 8: DISPLAY FINAL RESULTS
# ------------------------------------------------------------

final_df = pd.DataFrame(final_records)

print("\n" + "=" * 80)
print("FINAL TEMPLATE EVALUATION RESULTS")
print("=" * 80)

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", 180
):
    display(final_df)


# ------------------------------------------------------------
# STEP 9: CALCULATE FINAL CLASSIFICATION ACCURACY
# ------------------------------------------------------------

final_total = len(final_df)
final_correct = int(final_df["Correct"].sum())
final_valid = int(final_df["Valid Output"].sum())

final_accuracy = (
    final_correct / final_total * 100
    if final_total else 0
)

print("\n" + "=" * 80)
print("FINAL EVALUATION SUMMARY")
print("=" * 80)

print(f"Total new feedback items: {final_total}")
print(f"Correct classifications: {final_correct}/{final_total}")
print(f"Incorrect classifications: {final_total - final_correct}")
print(f"Final accuracy: {final_accuracy:.2f}%")
print(f"Valid responses: {final_valid}/{final_total}")
print(f"Invalid responses: {final_total - final_valid}")


# ------------------------------------------------------------
# STEP 10: REVIEW AMBIGUOUS OR DIFFICULT ITEMS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AMBIGUOUS FEEDBACK REVIEW")
print("=" * 80)

ambiguous_results = final_df[
    final_df["Difficulty"] == "Ambiguous"
]

display(
    ambiguous_results[
        [
            "Item",
            "Feedback",
            "Expected Category",
            "Predicted Category",
            "Justification",
            "Confidence",
            "Correct"
        ]
    ]
)

print(
    "\nReview whether each justification supports "
    "the chosen category and whether the model "
    "prioritized the customer's main concern."
)


# ------------------------------------------------------------
# STEP 11: DEMONSTRATE REUSABILITY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("REUSABLE TEMPLATE DEMONSTRATION")
print("=" * 80)

print(
    "The same function can classify new feedback "
    "without rewriting the prompt."
)

print(
    "\nExample usage:\n"
    "classify_customer_feedback("
    "'Please add a calendar integration.', "
    "use_examples=False)"
)


# ------------------------------------------------------------
# STEP 12: SAVE FINAL RESULTS
# ------------------------------------------------------------

final_df.to_csv(
    "part5_final_template_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print("- part5_final_template_results.csv")
print("\nPart 5 execution completed.")

print("""
Q9: Which prompting method or combination of methods would you use in the final application? Explain why it was the best fit for this task.
""")

print("\nAnswer 9")

print("""
For my final application, I would use a combination of zero-shot and reasoning-oriented prompting, with the option to add few-shot examples when 
needed. All three methods achieved 100% accuracy on the original 10 test items, so none performed better in classification accuracy. I chose 
reasoning-oriented instructions because they help the model focus on the customer's main concern and provide clear justifications. My final 
template also correctly classified all five new feedback items, including the two difficult examples, with no formatting errors. This approach 
works well because it is accurate on my test items, easy to reuse, and does not require examples for every classification.
""")

print("""
Q10: Describe the intentionally difficult test item you created. How did the model handle it, and what does that result reveal about the limits of 
your prompt template?
""")

print("\nAnswer 10")

print("""
One intentionally difficult test item I created was, "The reporting tool keeps crashing, and if this continues, we may cancel our subscription."
This was challenging because the feedback included both a technical problem and a possible cancellation. The model correctly classified it as 
CANCELLATION_RISK instead of TECHNICAL_ISSUE because the customer mentioned possibly canceling their subscription. It also provided a clear 
justification and high confidence. This shows that my template can handle some feedback that fits multiple categories by focusing on the 
customer's main concern. However, testing only five new items is not enough to prove that it will always make the correct decision. 
More complicated or unclear feedback could still lead to mistakes, so additional testing would be necessary.
""")

print("""
Q11: What changes would be needed to adapt your template to a different task, such as summarization, content generation, or question answering?
""")

print("\nAnswer 11")

print("""
To adapt my template to a different task, I would keep its reusable structure but change the instructions, category definitions, and output 
requirements. For example, if I wanted to use it for summarization, I would replace the classification categories with instructions to identify 
the main ideas and write a short, accurate summary. For content generation, I would include instructions about the topic, tone, audience, and 
writing style. For question answering, I would answer using the provided information and avoid making up details. I would also update the o
ptional examples to match the new task. The main benefit of my template is that I can change these sections without rebuilding the 
entire workflow.
""")


print("\n***********************End of Part 5*************************")



***********************Part 5: Reusable Prompt Template and Final Evaluation*************************


PART 5: FINAL REUSABLE PROMPT TEMPLATE
Model: llama3.2
Settings: {'temperature': 0, 'seed': 42, 'num_predict': 220}
Optional examples enabled: False
New test items: 5

Processing new feedback 1/5...
Feedback: I updated my billing address, but my latest invoice still shows the old address.
Difficulty: Standard
Expected: BILLING
Predicted: BILLING
Justification: Customer mentions an issue with their latest invoice
Confidence: high
Correct: True
--------------------------------------------------------------------------------

Processing new feedback 2/5...
Feedback: The search feature returns an error whenever I enter a customer's name.
Difficulty: Standard
Expected: TECHNICAL_ISSUE
Predicted: TECHNICAL_ISSUE
Justification: Error when using search feature
Confidence: high
Correct: True
--------------------------------------------------------------------------------

Processing new fee

,Item,Feedback,Difficulty,Expected Category,Predicted Category,Justification,Confidence,Correct,Valid Output,Formatting Issues
0,1,"I updated my billing address, but my latest invoice still shows the old address.",Standard,BILLING,BILLING,Customer mentions an issue with their latest invoice,high,True,True,None
1,2,The search feature returns an error whenever I enter a customer's name.,Standard,TECHNICAL_ISSUE,TECHNICAL_ISSUE,Error when using search feature,high,True,True,None
2,3,I really appreciate how easy the new onboarding process has become.,Standard,POSITIVE_FEEDBACK,POSITIVE_FEEDBACK,The customer expresses appreciation for the new onboarding process.,high,True,True,None
3,4,"The reporting tool keeps crashing, and if this continues, we may cancel our subscription.",Ambiguous,CANCELLATION_RISK,CANCELLATION_RISK,Customer mentions the tool crashing and considers canceling their subscription if the issue persists.,high,True,True,None
4,5,"The mobile app is useful, but it would be better if notifications could be customized.",Ambiguous,FEATURE_REQUEST,FEATURE_REQUEST,Customer requests a customization feature for notifications,high,True,True,None



FINAL EVALUATION SUMMARY
Total new feedback items: 5
Correct classifications: 5/5
Incorrect classifications: 0
Final accuracy: 100.00%
Valid responses: 5/5
Invalid responses: 0

AMBIGUOUS FEEDBACK REVIEW


,Item,Feedback,Expected Category,Predicted Category,Justification,Confidence,Correct
3,4,"The reporting tool keeps crashing, and if this...",CANCELLATION_RISK,CANCELLATION_RISK,Customer mentions the tool crashing and consid...,high,True
4,5,"The mobile app is useful, but it would be bett...",FEATURE_REQUEST,FEATURE_REQUEST,Customer requests a customization feature for ...,high,True



Review whether each justification supports the chosen category and whether the model prioritized the customer's main concern.

REUSABLE TEMPLATE DEMONSTRATION
The same function can classify new feedback without rewriting the prompt.

Example usage:
classify_customer_feedback('Please add a calendar integration.', use_examples=False)

Saved:
- part5_final_template_results.csv

Part 5 execution completed.

Q9: Which prompting method or combination of methods would you use in the final application? Explain why it was the best fit for this task.


Answer 9

For my final application, I would use a combination of zero-shot and reasoning-oriented prompting, with the option to add few-shot examples when 
needed. All three methods achieved 100% accuracy on the original 10 test items, so none performed better in classification accuracy. I chose 
reasoning-oriented instructions because they help the model focus on the customer's main concern and provide clear justifications. My final 
template al

# Part 6: AI Assistance Reflection

Because prompt engineering often involves using AI tools to brainstorm, revise, test, and critique prompts, you are expected to use an AI assistant during this assignment. You are also expected to evaluate and take responsibility for the suggestions it provides.



**Q12:** What AI assistant did you use during this assignment, and what parts of the work did it help with?

**A12:**



**Q13:** Describe one prompt-writing suggestion from the assistant that improved your results. What evidence suggests that it was useful?

**A13:**



**Q14:** Describe one suggestion from the assistant that was ineffective, unclear, or inappropriate for the task.

**A14:**



**Q15:** What did you need to test, revise, verify, or decide independently?

**A15:**

In [32]:

print("\n")
print("***********************Part 6: AI Assistance Reflection*************************")
print("\n")

print("""
Q12: What AI assistant did you use during this assignment, and what parts of the work did it help with?
""")

print("\nAnswer 12")

print("""
I used Claude as an AI assistant during this assignment to help me better understand zero-shot, few-shot, and reasoning-oriented prompting 
strategies. I reviewed the outputs, followed the assignment requirements, and used the evaluation results to draw my own conclusions
""")

print("""
Q13: Describe one prompt-writing suggestion from the assistant that improved your results. What evidence suggests that it was useful?
""")

print("\nAnswer 13")

print("""
One helpful suggestion from the AI assistant was to add clearer instructions for choosing between categories that overlap. For example, 
the reasoning-oriented prompt tells the model to consider cancellation risk when a customer mentions leaving the service. In my final evaluation, 
the model correctly classified a customer who reported a crashing tool and mentioned possibly canceling their subscription. The final template 
correctly classified all five new feedback items. Although the earlier prompting methods also achieved 100% accuracy, this result suggests 
that the decision process was useful for handling more complicated feedback.
""")

print("""
Q14: Describe one suggestion from the assistant that was ineffective, unclear, or inappropriate for the task.
""")

print("\nAnswer 14")

print("""
By including examples for each of the six categories to help the model understand the task. However, when I tested this approach, both zero-shot 
and few-shot prompting achieved 100% accuracy, and none of the classifications changed. This helped me understand that adding examples is not 
always necessary when the original prompt already provides clear instructions.
""")

print("""
Q15: What did you need to test, revise, verify, or decide independently?
""")

print("\nAnswer 15")

print("""
I was responsible for running the code, reviewing the model's responses, and checking whether the results matched the expected categories. 
I also examined the justifications, confidence ratings, and formatting to see whether the outputs made sense. I compared the three prompting 
methods using the same test items and reviewed the results from the final template. Based on those results, I decided that a reusable prompt 
with clear instructions and a decision process was a good fit for this task. I learned that AI suggestions can be helpful, but they still need 
to be tested and evaluated instead of accepted automatically.
""")

print("\n***********************End of Part 6*************************")
print("\n")





***********************Part 6: AI Assistance Reflection*************************



Q12: What AI assistant did you use during this assignment, and what parts of the work did it help with?


Answer 12

I used Claude as an AI assistant during this assignment to help me better understand zero-shot, few-shot, and reasoning-oriented prompting 
strategies. I reviewed the outputs, followed the assignment requirements, and used the evaluation results to draw my own conclusions


Q13: Describe one prompt-writing suggestion from the assistant that improved your results. What evidence suggests that it was useful?


Answer 13

One helpful suggestion from the AI assistant was to add clearer instructions for choosing between categories that overlap. For example, 
the reasoning-oriented prompt tells the model to consider cancellation risk when a customer mentions leaving the service. In my final evaluation, 
the model correctly classified a customer who reported a crashing tool and mentioned possib